In [2]:
from itertools import combinations
import numpy as np
import pandas as pd

In [37]:
mu = 3
delta_f = 0.1
delta_c = 0.1
n1 = 100
n2 = 100000
sigma1 = 1
sigma2 = 1
sigma3 = 1
rho = 0

M = np.array([[1, 0 , 0], [1, 1, 0], [1, 0, 1]])

def generate_Y(n1, n2, mu, delta_f, delta_c,
               sigma1, sigma2, sigma3, rho):

    # Generate paired Y1, Y2 observations
    cov_12 = np.array([
        [sigma1**2, rho * sigma1 * sigma2],
        [rho * sigma1 * sigma2, sigma2**2]
    ])

    Y12 = np.random.multivariate_normal(
        mean=[mu, mu + delta_f],
        cov=cov_12,
        size=n1
    )

    # Generate independent Y3 observations
    Y3 = np.random.normal(
        loc=mu + delta_c,
        scale=sigma3,
        size=n2
    )

    # Sample means
    Y_bar = np.array([
        Y12[:, 0].mean(),
        Y12[:, 1].mean(),
        Y3.mean()
    ])

    # Sample covariance estimates
    S12 = np.cov(Y12, rowvar=False, ddof=1)
    S3 = np.var(Y3, ddof=1)

    # Estimated covariance of the sample means
    Sigma_hat = np.array([
        [S12[0, 0] / n1, S12[0, 1] / n1, 0],
        [S12[1, 0] / n1, S12[1, 1] / n1, 0],
        [0,                   0,            S3 / n2]
    ])

    return Y_bar, Sigma_hat

In [38]:
def get_EB_parameters(Y, n1, n2, sigma1, sigma2, sigma3, rho):
    eps = 1e-6

    Sigma = np.array([
        [sigma1**2 / n1, rho * sigma1 * sigma2 / n1, 0],
        [rho * sigma1 * sigma2 / n1, sigma2**2 / n1, 0],
        [0, 0, sigma3**2 / n2]
    ])

    Sigma_inv = np.linalg.inv(Sigma)

    sigma2_delta_f = max(
        (Y[1] - Y[0])**2
        - (sigma1**2 + sigma2**2 - 2 * rho * sigma1 * sigma2) / n1,
        eps
    )

    sigma2_delta_c = max(
        (Y[2] - Y[0])**2
        - (sigma1**2 / n1 + sigma3**2 / n2),
        eps
    )

    prior_precision = np.diag([
        0,
        1 / sigma2_delta_f,
        1 / sigma2_delta_c
    ])

    V_post = np.linalg.inv(
        M.T @ Sigma_inv @ M + prior_precision
    )

    theta_post = (
        V_post
        @ M.T
        @ Sigma_inv
        @ Y
    )

    return theta_post, V_post

def ppi(y: np.ndarray, lam: float = 1.0) -> float:
    """The plug-in PPI estimator ``Y1 - Y2 + Y3`` (or arbitrary lambda)."""
    return float(y[0] + lam * (y[2] - y[1]))


def ppi_plus_plus(
    y: np.ndarray,
    n1: int,
    n2: int,
    cov_y: np.ndarray,
    lam: float | None = None,
) -> float:

    if n1 < 2 or n2 < 2:
        raise ValueError("n1 and n2 must both be at least 2")

    f_bar = (n1 * y[1] + n2 * y[2]) / (n1 + n2)

    # Recover observation-level variances
    var_f1 = n1 * cov_y[1, 1]
    var_f2 = n2 * cov_y[2, 2]

    # Pooled variance of all n1 + n2 predictions
    var_f = (
        (n1 - 1) * var_f1
        + (n2 - 1) * var_f2
        + n1 * (y[1] - f_bar) ** 2
        + n2 * (y[2] - f_bar) ** 2
    ) / (n1 + n2 - 1)

    var_f = max(float(var_f), np.finfo(float).eps)

    if lam is None:
        # Recover observation-level Cov(Y, f(X))
        cov_yf = n1 * cov_y[0, 1]

        lam = float(
            cov_yf / ((1 + n1 / n2) * var_f)
        )

    return ppi(y, lam)


In [39]:
B = 5_000

mu_hats = []

for b in range(B):
    Y, Sigma_hat = generate_Y(n1, n2, mu, delta_f, delta_c, sigma1, sigma2, sigma3, rho)
    theta_post, V_post = get_EB_parameters(
        Y, n1, n2, sigma1, sigma2, sigma3, rho
    )

    mu_hat_standard = Y[0]

    mu_hat_eb = theta_post[0]
    mu_hat_ppi = (Y[0] - Y[1] + Y[2])
    mu_hat_ppi_pp = ppi_plus_plus(Y, n1, n2, Sigma_hat)
    mu_hats.append((mu_hat_standard, mu_hat_eb, mu_hat_ppi, mu_hat_ppi_pp))

mu_hats = np.array(mu_hats)

In [40]:
mu_hat_standard = mu_hats[:, 0]
mu_hat_eb = mu_hats[:, 1]
mu_hat_ppi = mu_hats[:, 2]
mu_hat_ppi_pp = mu_hats[:, 3]

# standard
bias_standard = np.mean(mu_hat_standard) - mu
var_standard = np.var(mu_hat_standard)
mse_standard = np.mean((mu_hat_standard - mu)**2)

# EB
bias_eb = np.mean(mu_hat_eb) - mu
var_eb = np.var(mu_hat_eb)
mse_eb = np.mean((mu_hat_eb - mu)**2)

# PPI
bias_ppi = np.mean(mu_hat_ppi) - mu
var_ppi = np.var(mu_hat_ppi)
mse_ppi = np.mean((mu_hat_ppi - mu)**2)

#PPI ++
bias_ppi_pp = np.mean(mu_hat_ppi_pp) - mu
var_ppi_pp = np.var(mu_hat_ppi_pp)
mse_ppi_pp = np.mean((mu_hat_ppi_pp - mu)**2)

print("Standard")
print(f"  Bias:      {bias_standard:.6f}")
print(f"  Bias^2:    {bias_standard**2:.6f}")
print(f"  Variance:  {var_standard:.6f}")
print(f"  MSE:       {mse_standard:.6f}")

print("\nEB")
print(f"  Bias:      {bias_eb:.6f}")
print(f"  Bias^2:    {bias_eb**2:.6f}")
print(f"  Variance:  {var_eb:.6f}")
print(f"  MSE:       {mse_eb:.6f}")

print("\nPPI")
print(f"  Bias:      {bias_ppi:.6f}")
print(f"  Bias^2:    {bias_ppi**2:.6f}")
print(f"  Variance:  {var_ppi:.6f}")
print(f"  MSE:       {mse_ppi:.6f}")

print("\nPPI++")
print(f"  Bias:      {bias_ppi_pp:.6f}")
print(f"  Bias^2:    {bias_ppi_pp**2:.6f}")
print(f"  Variance:  {var_ppi_pp:.6f}")
print(f"  MSE:       {mse_ppi_pp:.6f}")


Standard
  Bias:      -0.002161
  Bias^2:    0.000005
  Variance:  0.010206
  MSE:       0.010211

EB
  Bias:      0.047453
  Bias^2:    0.002252
  Variance:  0.005622
  MSE:       0.007874

PPI
  Bias:      -0.000171
  Bias^2:    0.000000
  Variance:  0.020547
  MSE:       0.020547

PPI++
  Bias:      -0.002127
  Bias^2:    0.000005
  Variance:  0.010313
  MSE:       0.010317
